# Inference

Load one trained fold and complete a defective skull: 4096 points in, 6144 out.

Needs a GPU and the checkpoint from that fold. Metrics are not computed here —
that is `MSN_compare_runs.ipynb`.

## 1 · Setup

In [ ]:
import os
import sys
import time

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
for sub in ("src/models", "src/eval", "src/data"):
    sys.path.insert(0, os.path.join(REPO, sub))

import numpy as np
import paths
import report as rp

RUN = "msn_skullfix/cd_rep05_full_f0"   # fold 0 because it is fold 0, not because it looks best
DEVICE = "/GPU:0"

run = rp.Run(REPO, RUN)
cache = np.load(os.path.join(REPO, paths.DATA_CACHE))
ids, inputs, gt, scale_mm = cache["ids"], cache["inputs"], cache["gt"], cache["scale_mm"]

val = run.meta["val_ids"]
print(f"{run.label}: {run.config_str()}")
print(f"held out {len(val)} skulls, first is {val[0]}")

## 2 · Build the model and load the weights

The architecture is rebuilt from the run's own `run.json`, not assumed, because
some runs used a different topology and the weights would not fit.

Loading is checked. `load_weights` with `by_name` and `skip_mismatch` returns
without error while matching only part of the network, so a mismatch shows up as
bad predictions rather than an exception. Loading strictly and confirming the
tensors changed is what rules that out.

In [ ]:
import tensorflow as tf
for g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
import msn_skullfix as msn

cfg = rp.arch_config(msn, run.arch_key)
with tf.device(DEVICE):
    model = msn.build_model(cfg)
    before = [w.numpy().copy() for w in model.weights[:40]]
    model.load_weights(run.weights)                      # strict
changed = sum(1 for b, w in zip(before, model.weights) if not np.array_equal(b, w.numpy()))
assert changed > 30, f"only {changed}/40 tensors changed -- wrong architecture for these weights"
print(f"{model.count_params()/1e6:.1f}M parameters, {changed}/40 leading tensors overwritten")

text = np.load(os.path.join(REPO, paths.BERT_CACHE)) if cfg.use_text else None
def complete(k):
    x = [inputs[k][None]] + ([text[None]] if cfg.use_text else [])
    with tf.device(DEVICE):
        return model.predict(x, batch_size=1, verbose=0)[0]

## 3 · One skull

The text branch is a constant here. This dataset has one class, so the frozen
BERT output is the same vector for every sample and is read from cache rather
than recomputed.

In [ ]:
SKULL = val[0]
k = int(np.where(ids == SKULL)[0][0])

t0 = time.time()
pred = complete(k)
print(f"skull {SKULL}: {inputs[k].shape[0]} points in, {pred.shape[0]} out, {time.time() - t0:.1f}s")
print(f"scale {scale_mm[k]:.1f} mm -- multiply normalised distances by this for millimetres")

## 4 · Look at it

In [ ]:
import plotly.graph_objects as go

mask = np.load(os.path.join(REPO, "experiments_log", "defect_mask_labels.npz"))[SKULL]

fig = go.Figure()
for pts, name, colour, size in [
        (inputs[k],   "input (defective)",  "#B0BEC5", 1.3),
        (pred,        "completion",         "#1565C0", 1.6),
        (gt[k][mask], "defect ground truth", "#D32F2F", 2.4)]:
    fig.add_scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                      marker=dict(size=size, color=colour, opacity=0.6), name=name)
fig.update_layout(title=f"skull {SKULL} — grey is given, red is what had to be invented",
                  scene=dict(aspectmode="data"), height=620,
                  margin=dict(l=0, r=0, b=0, t=40)).show()

## 5 · The whole held-out set

Twenty skulls, the ones this fold never trained on. This is the input to
evaluation.

Inference is deterministic: validation sampling uses a fixed seed and is
stateless, so re-running gives bit-identical output. Training is not — same seed,
same config, and two runs diverge from the first epoch on GPU.

In [ ]:
sel = [int(np.where(ids == s)[0][0]) for s in val]

t0 = time.time()
preds = np.stack([complete(k) for k in sel])
print(f"{len(sel)} skulls in {time.time() - t0:.0f}s -> {preds.shape}")

a, b = complete(sel[0]), complete(sel[0])
print(f"re-running one skull: max difference {np.abs(a - b).max():.3e}")

## 6 · Save (optional)

Only useful for looking at the output offline. Nothing downstream reads this file
— evaluation recomputes predictions from the checkpoint, which is why deleting a
checkpoint is the irreversible step, not deleting this.

In [ ]:
out = os.path.join(REPO, "reports", "preview", f"{run.label}_preds.npz")
os.makedirs(os.path.dirname(out), exist_ok=True)
np.savez_compressed(out, ids=np.array(val), preds=preds.astype(np.float32),
                    scale_mm=scale_mm[sel])
print(f"{os.path.getsize(out)/1e6:.1f} MB -> {os.path.relpath(out, REPO)}")